# [1교시]

In [13]:
# 프롬프트 엔지니어링, 파인튜닝의 차이
# zero-shot prompting, few-shot prompting, 전체 매개변수 미세조절(full FIne Turning)
# 성능(accuracy) 자원소모(학습시간, 파라메터 업데이트 비율)

In [14]:
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devices : {device}')

using devices : cpu


### 데이터셋

In [15]:
train_data = [
    {"text": "This movie was absolutely fantastic! The acting was superb.", "label": "positive"},
    {"text": "I wasted two hours of my life. Terrible plot and cheap CGI.", "label": "negative"},
    {"text": "A true masterpiece of cinema. Highly recommend it to everyone.", "label": "positive"},
    {"text": "Extremely boring and predictable. I fell asleep halfway through.", "label": "negative"},
    {"text": "The cinematography was beautiful, and the story was touching.", "label": "positive"},
    {"text": "Horrible acting and bad direction. Do not watch this film.", "label": "negative"},
    {"text": "Brilliant performance by the lead actor, a must-watch.", "label": "positive"},
    {"text": "The plot made no sense, and the characters were annoying.", "label": "negative"},
    {"text": "I loved the soundtrack and the emotional depth of this movie.", "label": "positive"},
    {"text": "Total waste of money. I regret watching this garbage.", "label": "negative"},
    {"text": "Very engaging storyline with great character development.", "label": "positive"},
    {"text": "Slow, dull, and lacks any real substance.", "label": "negative"},
    {"text": "An amazing adventure that kept me on the edge of my seat.", "label": "positive"},
    {"text": "A complete disappointment. The trailer was much better than the film.", "label": "negative"},
    {"text": "Wonderfully written and beautifully executed. A delight to watch.", "label": "positive"}
]

test_data = [
    {"text": "The movie was mediocre, but the ending was spectacular!", "label": "positive"},
    {"text": "Worst movie I have seen this year. Avoid at all costs.", "label": "negative"},
    {"text": "A refreshing and delightful comedy that had me laughing all night.", "label": "positive"},
    {"text": "The acting was flat and the script felt extremely forced.", "label": "negative"},
    {"text": "An absolute gem of a film with outstanding visuals.", "label": "positive"}
]

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size: {len(test_data)}")

Train dataset size: 15
Test dataset size: 5


## 모델 로드 및 모델 매개변수(Parameter) 분석
Hugging Face의 `transformers` 라이브러리를 활용하여 구글이 공개한 지시어 미세조정(Instruction-tuned) 모델인 `google/flan-t5-small`을 로드합니다.
또한 파인튜닝 전에 모델의 **총 매개변수(Total Parameters)** 와 **학습 가능한 매개변수(Trainable Parameters)** 를 확인합니다.

In [16]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 7408.43it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [17]:
# _, param = next(iter(model.named_parameters()))
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)

In [18]:
# 추론 수행 함수
def generate_prediction(prompt, model, tokenizer,max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
    return pred_text
# 평가 루프
def evaluate_model(prompt, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0; results=[]
    # few shot 예시
    few_shot_examples = (
        "Review: The cinematography was beautiful, and the story was touching.\nSentiment:positive\n\n"
        "Review: Horrible acting and bad direction. Do not watch this film.\nSentiment:negative\n\n"
        "Review: Brilliant performance by the lead actor, a must-watch.\nSentiment:positive\n\n"
    )
    for item in test_data:
        text = item['text']
        true_label = item['label']
        if prompt_type =='zero_shot':
            prompt=f'Review:{text}\nSentiment: Answer with either positive or negative."'
        elif prompt_type =='few_shot':
            prompt=few_shot_examples + f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        else:
            prompt=f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        pred_label = generate_prediction(prompt,model,tokenizer)
        # 긍정/부정 단어 클랜징
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label
        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1
        results.append({
            'Text':text, 'True Label' : true_label, 
            'Pred Label':pred_label, 'Cleaned Pred' : cleaned_pred, 'Correct':is_correct
        })
    accuracy = correct / len(test_data)
    return accuracy, pd.DataFrame(results)

# [2교시]

## Zero Shot Prompting

In [19]:
zero_shot_acc, zero_shot_df = evaluate_model(model, tokenizer, test_data, 'zero_shot') 
print(f'zero shot acc : {zero_shot_acc}')
zero_shot_df

zero shot acc : 1.0


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,positive,positive,True
1,Worst movie I have seen this year. Avoid at al...,negative,negative,negative,True
2,A refreshing and delightful comedy that had me...,positive,positive,positive,True
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,positive,positive,True


## Few-Shot Prompting

In [20]:
fes_shot_acc, fes_shot_df = evaluate_model(model, tokenizer, test_data, 'few_shot')
print(f'fes shot acc : {fes_shot_acc}')
fes_shot_df

fes shot acc : 1.0


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,"The movie was mediocre, but the ending was spe...",positive,positive,positive,True
1,Worst movie I have seen this year. Avoid at al...,negative,negative,negative,True
2,A refreshing and delightful comedy that had me...,positive,positive,positive,True
3,The acting was flat and the script felt extrem...,negative,negative,negative,True
4,An absolute gem of a film with outstanding vis...,positive,positive,positive,True


## 전체 매개변수 파인튜닝(Full Fine Tuning)

In [23]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_function(examples):
    # T5 모델 입력 형식으로 문장들을 프롬프트 형태로 변환
    # batched=True로 map을 실행하므로 examples['text']는 문자열 1개가 아니라 문자열 리스트다.
    inputs = [
        f"Review: {text}\nSentiment: Answer with either positive or negative."
        for text in examples['text']
    ]

    # 입력 문장을 토큰화한다.
    # truncation=True: 너무 긴 문장은 max_length 기준으로 잘라서 길이 초과를 방지한다.
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    # 정답 라벨 positive / negative도 T5가 생성해야 하는 target 문장으로 토큰화한다.
    labels = tokenizer(
        text_target=examples['label'],
        max_length=8,
        truncation=True
    )

    # Seq2SeqTrainer가 학습할 정답 토큰을 labels 컬럼에 저장한다.
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


# 2. DataSet 객체 생성
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)


# 3. 토큰화 매핑
# 핵심 수정:
# batch_size=True가 아니라 batched=True를 사용해야 한다.
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

# 토큰화 결과 확인
print(tokenized_train[0])

Map: 100%|██████████| 5/5 [00:00<00:00, 1661.64 examples/s]

{'input_ids': [4543, 10, 100, 1974, 47, 2997, 2723, 55, 37, 6922, 47, 7857, 5, 4892, 2998, 295, 10, 11801, 28, 893, 1465, 42, 2841, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [1465, 1]}


In [24]:
# training Arguments 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_sentiment_results',
    eval_strategy='epoch',
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=2
)

# Data Collator, Trainer 객체
# DataCollatorForSeq2Seq가 배치마다 padding을 자동으로 맞춰준다.
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator
)

# 학습시간 측정
start_time = time.time()
trainer.train()
training_time = time.time() - start_time
print(f'Fine-tuning completed : {training_time:.2f} seconds')

Epoch,Training Loss,Validation Loss
1,0.123075,0.015057
2,0.002622,0.006753
3,0.000927,0.003861
4,0.000124,0.002841
5,0.000479,0.002552


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.13it/s]
c:\miniconda\envs\edu_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-tuning completed : 8.30 seconds


## 파인튜닝된 모델 최종 평가

In [26]:
ft_acc, ft_df = evaluate_model(model, tokenizer, test_data, "fine_tuned")
print(f"Fine-tuned Model Accuracy: {ft_acc:.2%}")
print(ft_df[["Text", "True Label", "Pred Label", "Correct"]].to_string())

Fine-tuned Model Accuracy: 100.00%
                                                                 Text True Label Pred Label  Correct
0             The movie was mediocre, but the ending was spectacular!   positive   positive     True
1              Worst movie I have seen this year. Avoid at all costs.   negative   negative     True
2  A refreshing and delightful comedy that had me laughing all night.   positive   positive     True
3           The acting was flat and the script felt extremely forced.   negative   negative     True
4                 An absolute gem of a film with outstanding visuals.   positive   positive     True


# [3교시]
- kaggle 감성분석 데이터셋

In [27]:
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


In [ ]:
!pip install kagglehub

In [49]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

from glob import glob
import pandas as pd
import re
df = pd.read_csv(glob(path+'/*')[0])
df['review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣a-zA-Z0-9\s]', '', x.strip().lower()))

df['sentiment'].apply(lambda x : x.strip().lower())
t = [{'text':review, 'label':sentiment} for review, sentiment in zip(df['review'].to_list(),df['sentiment'].to_list())]
train_data = t[:500]
test_data = t[-100:]
print(len(train_data), len(test_data))


Path to dataset files: C:\Users\최지용\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
500 100


In [50]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6763.17it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [51]:
# _, param = next(iter(model.named_parameters()))
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)

In [52]:
# 추론 수행 함수
def generate_prediction(prompt, model, tokenizer,max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
    return pred_text
# 평가 루프
def evaluate_model(prompt, tokenizer, test_data, prompt_type='zero_shot'):
    correct = 0; results=[]
    # few shot 예시
    few_shot_examples = (
        "Review: The cinematography was beautiful, and the story was touching.\nSentiment:positive\n\n"
        "Review: Horrible acting and bad direction. Do not watch this film.\nSentiment:negative\n\n"
        "Review: Brilliant performance by the lead actor, a must-watch.\nSentiment:positive\n\n"
    )
    for item in test_data:
        text = item['text']
        true_label = item['label']
        if prompt_type =='zero_shot':
            prompt=f'Review:{text}\nSentiment: Answer with either positive or negative."'
        elif prompt_type =='few_shot':
            prompt=few_shot_examples + f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        else:
            prompt=f'Review:{text}\nSentiment:  Answer with either positive or negative."'
        pred_label = generate_prediction(prompt,model,tokenizer)
        # 긍정/부정 단어 클랜징
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label
        is_correct = (cleaned_pred == true_label)
        if is_correct:
            correct += 1
        results.append({
            'Text':text, 'True Label' : true_label, 
            'Pred Label':pred_label, 'Cleaned Pred' : cleaned_pred, 'Correct':is_correct
        })
    accuracy = correct / len(test_data)
    return accuracy, pd.DataFrame(results)

In [53]:
zero_shot_acc, zero_shot_df =  evaluate_model(model, tokenizer,test_data,'zero_shot')
print(f'zero shot acc : {zero_shot_acc}')
zero_shot_df

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (891 > 512). Running this sequence through the model will result in indexing errors


zero shot acc : 0.91


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,i had few problems with this film and i have h...,positive,positive,positive,True
1,this program is a lot of fun and the title son...,positive,positive,positive,True
2,greenaway seems to have a habit of trying deli...,negative,negative,negative,True
3,this is one of the most hateful and cruel movi...,negative,negative,negative,True
4,air bud 2 golden receiver is a very bad rehear...,negative,negative,negative,True
...,...,...,...,...,...
95,i thought this movie did a down right good job...,positive,positive,positive,True
96,bad plot bad dialogue bad acting idiotic direc...,negative,negative,negative,True
97,i am a catholic taught in parochial elementary...,negative,negative,negative,True
98,im going to have to disagree with the previous...,negative,negative,negative,True


In [54]:
fes_shot_acc, fes_shot_df =  evaluate_model(model, tokenizer,test_data,'few_shot')
print(f'fes shot acc : {fes_shot_acc}')
fes_shot_df

fes shot acc : 0.9


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,i had few problems with this film and i have h...,positive,positive,positive,True
1,this program is a lot of fun and the title son...,positive,positive,positive,True
2,greenaway seems to have a habit of trying deli...,negative,negative,negative,True
3,this is one of the most hateful and cruel movi...,negative,negative,negative,True
4,air bud 2 golden receiver is a very bad rehear...,negative,negative,negative,True
...,...,...,...,...,...
95,i thought this movie did a down right good job...,positive,positive,positive,True
96,bad plot bad dialogue bad acting idiotic direc...,negative,negative,negative,True
97,i am a catholic taught in parochial elementary...,negative,negative,negative,True
98,im going to have to disagree with the previous...,negative,negative,negative,True


In [55]:
# 1. 데이터 토크나이징 함수 - 전처리
def preprocess_fuction(example):
    # T5모델에 맞게 토큰화
    inputs = [ f"Review: {text}\nSentiment: Answer with either positive or negative."  for text in example['text'] ]
    model_input = tokenizer(inputs,max_length=128, truncation=True)
    # 라벨을 토큰화
    lables = tokenizer(text_target=example['label'],max_length=128, truncation=True)
    model_input['labels'] = lables['input_ids']
    return model_input
# 2. DataSet객체
from datasets import Dataset
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 3. 토큰화 매핑()
tokenized_train = train_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_fuction,batched=True, remove_columns=train_dataset.column_names)

Map: 100%|██████████| 100/100 [00:00<00:00, 1598.01 examples/s]


In [56]:
# training Arguments 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_sentiment_results',
    eval_strategy='epoch',
    learning_rate=3e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    logging_steps=2
)
# Data Collator, Trainer 객체
data_collator = DataCollatorForSeq2Seq(tokenizer,model=model)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,train_dataset=tokenized_train, eval_dataset=tokenized_test,
    processing_class=tokenizer, data_collator = data_collator
)
# 학습시간 측정
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'Fine-turning completed : {training_time:.2f} seconds') 

c:\miniconda\envs\edu_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.181110,0.197561
2,0.127068,0.236605
3,0.279355,0.378521
4,0.001545,0.402781
5,0.192575,0.433467


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.07it/s]
c:\miniconda\envs\edu_env\lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Fine-turning completed : 743.15 seconds


In [57]:
ft_acc, ft_df = evaluate_model(model, tokenizer, test_data, "fine_tuned")
print(f"Fine-tuned Model Accuracy: {ft_acc:.2%}")
print(ft_df[["Text", "True Label", "Pred Label", "Correct"]].to_string())

Fine-tuned Model Accuracy: 89.00%
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

# [4교시]

첫번째 실습에서 우리는 풀 파인튜닝(Full Fine-tuning)이 뛰어난 정확도를 내지만, 모델의 모든 매개변수(100%)를 학습해야 하므로 컴퓨팅 자원과 저장공간 소모가 극심함을 배웠습니다.

이번 2번재 실습에서는 이러한 문제를 극복하기 위한 대표적인 **PEFT (Parameter-Efficient Fine-Tuning)** 기법인 **LoRA (Low-Rank Adaptation)** 의 개념을 배우고, 직접 모델에 적용하여 학습을 수행해 봅니다.

## 1. PEFT와 LoRA의 이해

### 1) PEFT (Parameter-Efficient Fine-Tuning) 란?
- 사전 훈련된 거대 모델(Base Model)의 대부분 가중치는 **동결(Freeze)** 하고, 매우 일부분의 매개변수만 추가하거나 튜닝하여 새로운 작업에 적응시키는 기법입니다.
- **장점**:
  - **VRAM 소모 급감**: 최적화 기구(Adam 등)가 관리해야 할 파라미터가 크게 줄어 GPU 메모리를 훨씬 적게 씁니다.
  - **저장용량 절약**: 전체 모델 파일(예: 300MB~수십GB)을 태스크마다 따로 저장할 필요 없이, 몇십 KB ~ 몇 MB 크기의 가중치(어댑터) 파일만 관리하면 됩니다.
  - **치명적 망각 방지**: 기존 파라미터를 그대로 보존하므로 일반 성능이 유지됩니다.

### 2) LoRA (Low-Rank Adaptation) 의 작동 원리
- 모델이 학습할 가중치 업데이트 행렬 $\Delta W$를 직접 훈련하는 대신, 저차원의 두 개의 행렬 $A$와 $B$의 곱으로 **분해(Decomposition)** 하여 학습시키는 기법입니다.
  $$\Delta W \approx B \cdot A$$
  - 여기서 원래 가중치가 $d \times k$ 차원이고 저차원 랭크가 $r$ ($r \ll d, k$) 일 때:
    - $B \in \mathbb{R}^{d \times r}$ (0으로 초기화)
    - $A \in \mathbb{R}^{r \times k}$ (가우시안 분포로 초기화)
- 결과적으로 가중치 매개변수의 수는 $(d \times k)$ 에서 $r \times (d + k)$ 로 압도적으로 줄어듭니다.

## 라이브러리

In [5]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

c:\miniconda\envs\edu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\miniconda\envs\edu_env\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


using devcies : cpu


## 데이터

In [6]:
# 1교시에 사용한 데이터 train, test
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

from glob import glob
import pandas as pd
import re
df = pd.read_csv(glob(path+'/*')[0])
df['review'] = df['review'].apply(lambda x : re.sub(r'[^가-힣a-zA-Z0-9\s]', '', x.strip().lower()))

df['sentiment'].apply(lambda x : x.strip().lower())
t = [{'text':review, 'label':sentiment} for review, sentiment in zip(df['review'].to_list(),df['sentiment'].to_list())]
train_data = t[:500]
test_data = t[-100:]
print(len(train_data), len(test_data))

Path to dataset files: C:\Users\최지용\.cache\kagglehub\datasets\lakshmi25npathi\imdb-dataset-of-50k-movie-reviews\versions\1
500 100


# [5교시]

## 베이스 모델 로드 및 LoRA 설정 적용
- `peft` 라이브러리의 `LoraConfig`를 선언하고 모델에 주입합니다.
- `target_modules` 인자에 튜닝을 진행할 어텐션 매개변수인 쿼리(`q`)와 밸류(`v`) 행렬을 명시합니다.

In [7]:
# 사용할 모델 이름 설정
model_name = 'google/flan-t5-small'

# 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Seq2Seq 모델 불러오기
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 모델을 GPU 또는 CPU로 이동
model.to(device)

# LoRA 설정
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # T5는 Seq2Seq 모델이므로 SEQ_2_SEQ_LM 사용
    r=8,                              # LoRA 저차원 랭크 크기
    lora_alpha=32,                    # LoRA 가중치 스케일 값
    lora_dropout=0.1,                 # 과적합 방지를 위한 dropout
    target_modules=['q', 'v']         # 오류 원인: target_modules가 두 번 있었음. 하나만 남겨야 함
)

# 기존 모델에 LoRA 어댑터를 붙여 PEFT 모델로 변환
model = get_peft_model(model, peft_config)

# 학습 가능한 파라미터 개수를 계산하는 함수
def get_trainable_params(model):
    all_param = 0
    trainable_params = 0

    # 모델 안의 모든 파라미터를 하나씩 확인
    for _, param in model.named_parameters():
        all_param += param.numel()  # 전체 파라미터 수 누적

        # requires_grad=True인 파라미터만 실제 학습 대상
        if param.requires_grad:
            trainable_params += param.numel()

    # 전체 파라미터, 학습 파라미터, 학습 비율 반환
    return all_param, trainable_params, trainable_params / all_param * 100

# 파라미터 개수 확인
all_p, train_p, pct = get_trainable_params(model)

# 결과 출력
all_p, train_p, pct

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3730.66it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


(77305216, 344064, 0.445072166928555)

### 데이터 토큰화 처리

In [8]:
# Hugging Face Dataset 객체 사용
from datasets import Dataset

# 데이터 토큰화 함수
def preprocess_function(examples):
    # T5 모델 입력 형식으로 문장을 변환
    inputs = [
        f"Review: {text}\nSentiment: Answer with either positive or negative."
        for text in examples['text']
    ]

    # 입력 문장 토큰화
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True
    )

    # 정답 라벨 토큰화
    labels = tokenizer(
        text_target=examples['label'],
        max_length=8,
        truncation=True
    )

    # Trainer가 사용할 정답 토큰 저장
    model_inputs['labels'] = labels['input_ids']

    return model_inputs

# 기존 train_data, test_data를 Dataset 형식으로 변환
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

# 학습 데이터 토큰화
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

# 테스트 데이터 토큰화
# 오류 원인: 기존 코드에서 test_dataset이 아니라 train_dataset.column_names를 사용했음
tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

Map: 100%|██████████| 100/100 [00:00<00:00, 1472.59 examples/s]


### LoRa Fine-tuning 학습

In [9]:
# 학습 설정
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5_lora_results',   # 학습 결과가 저장될 폴더 이름

    eval_strategy='epoch',            # 매 epoch가 끝날 때마다 평가 진행

    learning_rate=1e-4,               # LoRA 학습률 설정

    num_train_epochs=2,               # 전체 데이터를 2번 반복해서 학습

    predict_with_generate=True,       # T5 모델이 정답을 생성하는 방식으로 평가

    logging_steps=2,                  # 2 step마다 학습 로그 출력

    save_strategy='no',               # Trainer가 자동으로 체크포인트 저장하지 않게 설정
                                      # 이전에 T5 공유 임베딩 저장 에러가 났기 때문에 자동 저장을 끄는 방식으로 해결

    dataloader_pin_memory=False       # CPU 환경에서 pin_memory 경고가 나오지 않도록 설정
)

# 배치마다 길이가 다른 문장들을 자동으로 padding 처리해주는 collator 생성
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,              # 현재 사용하는 T5 tokenizer
    model=model                       # LoRA가 적용된 T5 모델
)

# Seq2SeqTrainer 객체 생성
trainer = Seq2SeqTrainer(
    model=model,                      # 학습할 LoRA 적용 모델

    args=training_args,               # 위에서 설정한 학습 옵션

    train_dataset=tokenized_train,    # 토큰화된 학습 데이터

    eval_dataset=tokenized_test,      # 토큰화된 평가 데이터

    processing_class=tokenizer,       # tokenizer를 Trainer에 전달

    data_collator=data_collator       # 배치 padding 처리를 담당하는 collator
)

# 학습 시작 시간 기록
start_time = time.time()

# LoRA 학습 실행
trainer.train()

# 학습에 걸린 시간 계산
training_time = time.time() - start_time

# 학습 완료 메시지 출력
print(f'Fine-tuning completed : {training_time:.2f} seconds')

Epoch,Training Loss,Validation Loss
1,3.022674,2.276725
2,1.716372,1.373289


Fine-tuning completed : 249.94 seconds


In [10]:
# 현재 노트북 커널 기준으로 패키지를 설치하기 위한 sys import
import sys

# trl이 현재 커널에 없으면 설치
# 터미널 pip install과 노트북 커널 환경이 다를 수 있어서 sys.executable 기준으로 설치
try:
    from trl import DPOTrainer, DPOConfig
except ModuleNotFoundError:
    !{sys.executable} -m pip install trl peft accelerate
    from trl import DPOTrainer, DPOConfig

# DPO 실습에 필요한 라이브러리 import
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd

# transformers import
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,       # DPO는 GPT 계열 Causal LM 사용
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

# GPU 사용 가능 여부 확인
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 현재 사용 장치 출력
print(f'using device : {device}')

# 현재 파이썬 경로 확인
print(sys.executable)

using device : cpu
c:\miniconda\envs\edu_env\python.exe


### 평가

In [11]:
# 추론 수행 함수
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    # 입력 문장을 토큰화하고 현재 device로 이동
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    # 평가 단계에서는 gradient 계산이 필요 없음
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens
        )

    # 생성 결과를 문자열로 변환
    pred_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip().lower()

    return pred_text


# 평가 함수
def evaluate_model(model, tokenizer, test_data, prompt_type='fine_tuned'):
    correct = 0
    results = []

    # 테스트 데이터 하나씩 평가
    for item in test_data:
        text = item['text']
        true_label = item['label']

        # 학습 때 사용한 입력 형식과 동일하게 맞춤
        prompt = f"Review: {text}\nSentiment: Answer with either positive or negative."

        # 모델 예측
        pred_label = generate_prediction(prompt, model, tokenizer)

        # 예측 결과 정리
        if 'positive' in pred_label:
            cleaned_pred = 'positive'
        elif 'negative' in pred_label:
            cleaned_pred = 'negative'
        else:
            cleaned_pred = pred_label

        # 정답 여부 확인
        is_correct = cleaned_pred == true_label

        if is_correct:
            correct += 1

        # 결과 저장
        results.append({
            'Text': text,
            'True Label': true_label,
            'Pred Label': pred_label,
            'Cleaned Pred': cleaned_pred,
            'Correct': is_correct
        })

    # 정확도 계산
    accuracy = correct / len(test_data)

    return accuracy, pd.DataFrame(results)


# LoRA 파인튜닝 모델 평가
ft_acc, ft_df = evaluate_model(model, tokenizer, test_data, 'fine_tuned')

# 정확도 출력
print(f'LoRA Fine-tuned Accuracy : {ft_acc:.2%}')

# 결과 확인
ft_df

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (891 > 512). Running this sequence through the model will result in indexing errors


LoRA Fine-tuned Accuracy : 92.00%


,Text,True Label,Pred Label,Cleaned Pred,Correct
0,i had few problems with this film and i have h...,positive,positive,positive,True
1,this program is a lot of fun and the title son...,positive,positive,positive,True
2,greenaway seems to have a habit of trying deli...,negative,negative,negative,True
3,this is one of the most hateful and cruel movi...,negative,negative,negative,True
4,air bud 2 golden receiver is a very bad rehear...,negative,negative,negative,True
...,...,...,...,...,...
95,i thought this movie did a down right good job...,positive,positive,positive,True
96,bad plot bad dialogue bad acting idiotic direc...,negative,negative,negative,True
97,i am a catholic taught in parochial elementary...,negative,negative,negative,True
98,im going to have to disagree with the previous...,negative,negative,negative,True


# 1. 인간 피드백 기반 강화학습 (RLHF)과 DPO (Direct Preference Optimization)

이전 차시까지 우리는 입력문장에 부합하는 대답을 생성하도록 정밀 학습을 시키는 파인튜닝(Full FT, LoRA) 기법을 실습했습니다.
하지만 AI가 유용하고(Helpful), 정직하며(Honest), 무해한(Harmless) 답변을 제공하도록 즉, **인간의 가치와 선호에 정렬(Alignment)** 되도록 하는 것은 단순한 분류나 텍스트 복제 방식의 학습으로는 달성하기 어렵습니다.

 LLM의 정렬을 위해 필수적으로 쓰이는 **RLHF** 와 최신 정렬 기법인 **DPO** 의 개념을 이해하고, 실습용 `trl` 라이브러리를 활용하여 GPT-2 모델을 직접 선호 정렬 학습시킵니다.

## 1. RLHF와 DPO의 핵심 이해

### 1) RLHF (Reinforcement Learning from Human Feedback) 란?
- 모델의 출력을 인간의 평가 기준에 부합하도록 지도하는 3단계 정렬 파이프라인입니다.
  1. **SFT (Supervised Fine-tuning)**: 지시어와 원하는 고품질 답변 데이터로 지도학습을 진행합니다.
  2. **RM (Reward Model, 보상 모델) 학습**: 동일한 질문에 대한 여러 답변을 인간이 선호도에 따라 순위를 매기고, 이를 바탕으로 질문-답변의 점수를 예측하는 보상 모델을 학습시킵니다.
  3. **PPO (Proximal Policy Optimization) 강화학습**: 2단계에서 만든 보상 모델의 점수를 극대화하도록 PPO 알고리즘을 사용해 LLM의 가중치를 갱신합니다.
- **한계**: 보상 모델 학습 및 PPO 강화학습을 결합하는 루프가 복잡하고, 학습이 매우 불안정하며 메모리 소모가 극도로 심합니다.

### 2) DPO (Direct Preference Optimization) 란?
- Stanford 대학 연구진이 제안한 기법으로, **별도의 보상 모델(Reward Model)과 복잡한 강화학습 PPO 과정 없이** , 사람이 선호한 답변(Chosen)과 거부한 답변(Rejected)만을 가지고 **수학적 손실 함수를 사용해 직접 정책 모델을 튜닝** 하는 방식입니다.
- 수학적 치환을 통해 RLHF의 복잡한 PPO 목적 함수를 단순한 이진 교차 엔트로피(Binary Cross-Entropy) 손실 함수로 매핑하여 학습 안전성과 효율성을 대폭 끌어올렸습니다.

In [12]:
from datasets import Dataset
from trl import DPOTrainer,DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


## 2. DPO용 가상 선호 데이터셋 정의
DPO 학습 데이터셋은 반드시 아래 세 가지 열을 갖춰야 합니다:
1. `prompt`: 모델에 전달하는 질문/지시문
2. `chosen`: 사람이 선호하는 긍정 답변 (Target Winner)
3. `rejected`: 사람이 선호하지 않는 잘못되거나 부정적인 답변 (Target Loser)

In [13]:
dpo_train_data = [
    {
        "prompt": "Review: This movie is a masterpiece. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: I hated this film. Total waste of time. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    },
    {
        "prompt": "Review: Outstanding acting and brilliant plot. Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Terrible acting and boring screenplay. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"

    }
]

dpo_test_data = [
    {
        "prompt": "Review: The movie was mediocre, but the ending was spectacular! Sentiment: ",
        "chosen": "positive",
        "rejected": "negative"
    },
    {
        "prompt": "Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: ",
        "chosen": "negative",
        "rejected": "positive"
    }
]

train_dataset = Dataset.from_list(dpo_train_data)
test_dataset = Dataset.from_list(dpo_test_data)
print("DPO dataset initialized!")

DPO dataset initialized!


## 3. 베이스 모델 로드 및 LoRA 설정 적용
- DPO 학습 대상 모델로 디코더 형태의 모델인 `gpt2`를 로드합니다.
- 학습 파라미터 소모를 대폭 줄이기 위해 LoRA 기법을 결합하여 어댑터만 튜닝하도록 구성합니다.

### AutoModelForSeq2SeqLM, AutoModelForCausalLM 비교
- Encoder-Decoder               Decoder Only
-   T5, BART                    GPT,LIma
- 입력 -> Encoder                입력->Decoder 
- 벡터 -> Decoder- 출력           -> 다른 토큰으로 예측 -> 다른 토큰으로 예측

In [14]:
import torch
device = "cuda" if torch.cuda.is_available() else 'cpu'
model = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

# gpt-2 패딩 설정 조정
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id
model.to(device)

# LoRA 설정
peft_config= LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r = 8,          # 저차원 랭크의 크기
    lora_alpha=32,  # 가중치 스케일 상수
    lora_dropout=0.1,
    target_modules=['c_attn'],
)

# PEFT 모델로 변환
model = get_peft_model(model, peft_config=peft_config)
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4067.92it/s]
c:\miniconda\envs\edu_env\lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


(124734720, 294912, 0.23643136409814366)

# [6교시] ~ [7교시]

## 4. DPO 학습 설정 및 Trainer 구동
- `DPOConfig` 클래스를 선언하고 `DPOTrainer`에 로드하여 학습을 진행합니다.
- `beta`는 DPO의 중요 매개변수로, 기준 참조 모델(Reference Model)과의 KL 패널티 계수를 나타냅니다. (보통 0.1~0.5 사용)

In [15]:
train_args = DPOConfig(
    output_dir = './gpt_dpo_results',
    eval_strategy='epoch',
    learning_rate=5e-4,  # LoRA DPO용 다소 높은 학습률
    num_train_epochs=2,
    beta = 0.1,
    use_cpu = True
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,  # 메모리 보존
    args = train_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)
start_time = time.time() 
trainer.train()
training_time = time.time() - start_time
print(f'DPO training  completed : {training_time:.2f} seconds') 

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,3.006859,116.000000,-78.910980,-78.910980,0.000000,0.020498,0.020498,0.000000,0.000000,-6.988075,-6.988075
2,No log,0.693147,2.990977,232.000000,-79.050644,-79.050644,0.000000,0.009153,0.009153,0.000000,0.000000,-7.101522,-7.101522


DPO training  completed : 5.91 seconds


## 5. DPO 정렬 모델 결과 검증 및 평가
학습이 끝난 모델에 테스트용 리뷰 데이터를 주입해 보아 긍정/부정 선호 정렬이 올바르게 나타나는지 확인합니다.

In [16]:
def generate_prediction(prompt, model, tokenizer, max_new_tokens=10):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # 생성된 텍스트 중 프롬프트 부분을 제외한 새로 추가된 토큰만 해독
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()
    return pred_text

print("\n--- Testing DPO Aligned Model ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Testing DPO Aligned Model ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: i'm not sure if it's because of

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: i'm not a fan of the movie,



## Supervised Fine-Tuning(SFT)학습
- DPO 모델이 Review: ...... Sentiment:...... 뒤에 단답형 답변 positive negative를 붙여서 완료하는 규칙

In [17]:
sft_train_list = [
    {'text':f"{item['prompt']}{item['chosen']}{tokenizer.eos_token}"}
for item in dpo_train_data
]
sft_dataset = Dataset.from_list(sft_train_list)
from trl import SFTTrainer, SFTConfig
sft_config = SFTConfig(
    output_dir="./gpt2_sft_results",
    eval_strategy="no",
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    num_train_epochs=12,      # 데이터가 극소량이므로 형식을 인지하도록 에폭을 늘립니다.
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_dataset,
    processing_class=tokenizer
)

sft_start_time = time.time()
sft_trainer.train()
sft_time = time.time() - sft_start_time
print(f"SFT completed in {sft_time:.2f} seconds.")

Tokenizing train dataset: 100%|██████████| 4/4 [00:00<00:00, 1129.93 examples/s]
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.240132
20,3.971957


SFT completed in 11.16 seconds.


# [8교시]

- SFT를 통해 출력 형식을 터득한 모델 가중치를 기반으로, 긍정 평가는 긍정 단어로, 부정 평가는 부정 단어로 밀어내는 **선호 정렬 DPO 학습** 을 수행합니다.

In [20]:
training_args = DPOConfig(
    output_dir="./gpt2_dpo_results",
    eval_strategy="epoch",
    learning_rate=2e-4,       # SFT 모델을 기반으로 하므로 적절한 학습률 적용
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    beta=0.1,                 # DPO margin
    max_length=64,
    use_cpu=(device == "cpu"),
    report_to="none"
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,           # PEFT 활성 상태이므로 None으로 주면 베이스 모델(SFT 미수행)을 ref로 참조
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

dpo_start_time = time.time()
trainer.train()
dpo_time = time.time() - dpo_start_time
print(f"DPO completed in {dpo_time:.2f} seconds.")

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer

Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,No log,0.693147,3.441370,116.000000,-78.819206,-78.819206,0.000000,0.008688,0.008688,0.000000,0.000000,-6.629942,-6.629942
2,No log,0.693147,3.449598,232.000000,-79.183426,-79.183426,0.000000,0.011031,0.011031,0.000000,0.000000,-6.606512,-6.606512
3,No log,0.693147,3.389118,348.000000,-78.865738,-78.865738,0.000000,0.010503,0.010503,0.000000,0.000000,-6.611794,-6.611794


DPO completed in 7.52 seconds.


In [ ]:
print("\n--- Inference after DPO (Final Alignment Check) ---")
for item in dpo_test_data:
    prompt = item["prompt"]
    true_chosen = item["chosen"]
    pred = generate_prediction(prompt, model, tokenizer)
    print(f"Prompt: {prompt}")
    print(f"-> Expected (Chosen): {true_chosen} | Model Output: {pred}\n")


--- Inference after DPO (Final Alignment Check) ---
Prompt: Review: The movie was mediocre, but the ending was spectacular! Sentiment: 
-> Expected (Chosen): positive | Model Output: i'm not sure if it's because of

Prompt: Review: Worst movie I have seen this year. Avoid at all costs. Sentiment: 
-> Expected (Chosen): negative | Model Output: i'm not a fan of the movie.

